In [ ]:
import sys
sys.path.insert(1, '../utils/')

In [ ]:
import datacube

from IPython.display import HTML, display

from utils.deafrica_plotting import rgb
from utils.nostra_dc import get_product_bbox
from utils.nostra_mapping import create_map, last_drawn_rectangle, get_utm_epsg_code

In [ ]:
dc = datacube.Datacube(app="Test_MinIO_indexation")

In [ ]:
product = 'ls89_c2l2_sp_local'

In [ ]:
%%time

product_bbox = get_product_bbox(dc, product, split_size=-1, stability_threshold=5)
print(product_bbox.bounds)

In [ ]:
m = create_map(tuple(product_bbox.bounds))
display(m)

In [ ]:
aoi_poly = last_drawn_rectangle
if len(aoi_poly) == 0:
    # Change the background color of the output area to light red
    display(HTML('''
        <div style="background-color: red; padding: 10px; border-radius: 5px;">
            The area of interest polygon (aoi_poly) has not been created. Please draw it in the previous cell.
        </div>
    '''))
else:
    aoi_poly = aoi_poly.get('bounds')

In [ ]:
# get EPSG code for the center of the AoI

epsg_code = get_utm_epsg_code((aoi_poly[1] + aoi_poly[3]) / 2, (aoi_poly[0] + aoi_poly[2]) / 2)

In [ ]:
dss = dc.find_datasets(product=product,
                       x=(aoi_poly[0], aoi_poly[2]),
                       y=(aoi_poly[1], aoi_poly[3]))
times = [ds.time.begin for ds in dss]
start_date = min(times)
end_date = max(times)

In [ ]:
lazy_ds = dc.load(product=product,
                  measurements=['red', 'green', 'blue'],
                  output_crs=f"EPSG:{epsg_code}",
                  y=(aoi_poly[1], aoi_poly[3]),
                  x=(aoi_poly[0], aoi_poly[2]),
                  time=(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')),
                  resolution=30,
                  group_by="solar_day",
                  dask_chunks={},
                  )

In [ ]:
# customized chunking is less buggy when performed after loading !!!
lazy_ds = lazy_ds.chunk({"x": 512, "y": 512, "time": 1})
print(lazy_ds)

In [ ]:
rgb(lazy_ds, col="time", col_wrap=4, robust=True)